# Qwen3-8B distillation on a Colab L4

This run collects **1,000 new training pairs in 5 rounds**, with **120 general and 80 adaptive pairs** per round. It uses Unsloth with 4-bit QLoRA, training batch size 4 and inference batch size 8, 360 fixed development cases, and a final holdout. Training uses independently verified teacher answers. Student mistakes prioritize adaptive cases and guide later rounds.

1. Select **L4 GPU** in Colab and run the setup cells through **Load the student model**. Authorize Drive, upload the matching `colab_distillation_bundle.zip`, and supply `OPENAI_API_KEY` through Colab Secrets or the hidden prompt.
2. When ready, run **Start or resume training** and enter the total teacher API budget in USD. The API budget is separate from Colab compute units.
3. After a disconnect, use the same settings and Drive folder to resume.

Run files are saved under `MyDrive/HealthInsuranceSLM/colab_l4_balanced_5x200_v3`. The first setup may download the base model. A fresh run starts from the latest compatible GitHub release adapter. For a private repository, add `GITHUB_TOKEN` to Colab Secrets and enable notebook access.


In [ ]:
from pathlib import Path
import os
import sys

if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("Restart the Colab session before running this notebook from the top.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/hf-cache"
for name in ("LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"):
    os.environ.pop(name, None)
# This experiment saves to Drive, independently of the ongoing Kaggle run.
os.environ.pop("KAGGLE_ADAPTER_DATASET", None)
os.environ["GITHUB_ADAPTER_UPLOAD"] = "0"
os.environ["DISTILLATION_DRIVE_BACKUP"] = "1"
os.environ["DISTILLATION_LOCAL_LOCK_DIR"] = "/content/distillation-colab-locks"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/HealthInsuranceSLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive did not mount; do not run training on temporary storage.")
print("Persistent Google Drive folder:", DRIVE_ROOT)


## Upload the companion code bundle

The ZIP contains the current loop code and fixed evaluation corpus, not model weights or credentials. This avoids a stale GitHub checkout. Upload the ZIP supplied with this notebook; it is checked against the expected SHA256 before extraction. Extraction writes only into this experiment's `/content/Health-Insurance-SLM-Colab` code folder, not Google Drive or your Kaggle run.


In [ ]:
import hashlib
import zipfile

EXPECTED_BUNDLE_SHA256 = "5964bb0e45f39a1f771625261ae9fd3c8238242272f03858565eaa18253652a5"
BUNDLE = Path("/content/colab_distillation_bundle.zip")
PROJECT_ROOT = Path("/content/Health-Insurance-SLM-Colab")

if not BUNDLE.is_file() or hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
    from google.colab import files
    print("Upload colab_distillation_bundle.zip supplied with this notebook.")
    uploaded = files.upload()
    payload = next((data for data in uploaded.values()
                    if hashlib.sha256(data).hexdigest() == EXPECTED_BUNDLE_SHA256), None)
    if payload is None:
        raise RuntimeError("None of the uploaded files matches this notebook’s bundle checksum.")
    BUNDLE.write_bytes(payload)
    del uploaded, payload
with zipfile.ZipFile(BUNDLE) as bundle:
    for entry in bundle.infolist():
        destination = (Path("/content") / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT_ROOT):
            raise RuntimeError("Unexpected path inside the code bundle.")
        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError("Symbolic links are not allowed in the code bundle.")
    bundle.extractall("/content")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
REQUIREMENTS = str(PROJECT_ROOT / "requirements-colab.txt")
print("Code and fixed evaluation data ready:", PROJECT_ROOT)


In [ ]:
from importlib.metadata import version
import subprocess

# Preserve the CUDA-enabled PyTorch build supplied by Colab.
torch_constraint = Path("/content/distillation-torch-constraint.txt")
torch_constraint.write_text(f"torch=={version('torch')}\n")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", str(torch_constraint),
     "-r", REQUIREMENTS],
    check=True,
)


If Colab asks for a runtime restart after installation, restart it and rerun from the first setup cell. Your Drive run folder is preserved. The uploaded ZIP may need uploading again if the VM was replaced.


In [ ]:
import getpass

if not os.environ.get("OPENAI_API_KEY", "").strip():
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (hidden): ").strip()
if not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("OPENAI_API_KEY is required for teacher training-data generation.")
print("Teacher credential loaded. It will not be written to Google Drive or adapter archives.")


## Configure the run

A fresh run initializes a rank-32 rsLoRA adapter on Qwen3-8B. An existing run resumes from its Drive folder. This profile initializes its own adapter.


In [ ]:
import json
from self_instruct_pipeline.config import load_config
from self_instruct_pipeline.persistence import Store, atomic_json
from distillation_loop.core import load_loop_config
from distillation_loop.colab import distillation_1000_configs, save_colab_subset
from distillation_loop.preflight import readiness

RUN_NAME = "colab_l4_balanced_5x200_v3"
TRAIN_BATCH_SIZE = 4
INFERENCE_BATCH_SIZE = 8
MODEL_NAME = "Qwen/Qwen3-8B"
RUN_DIR = DRIVE_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
parent_config, loop_config = distillation_1000_configs(
    load_config(PROJECT_ROOT / "config/self_instruct.json"),
    load_loop_config(str(PROJECT_ROOT / "config/distillation_loop.json")),
    RUN_DIR,
)
# Preserve the experiment configuration used to identify existing Drive runs.
loop_config["training"].update(per_device_train_batch_size=2, gradient_accumulation_steps=2)
loop_config["inference"]["batch_size"] = 4
# Validate the profile without overwriting a previous run's configuration.
profile_path = Path("/content/requested_colab_config.json")
atomic_json(profile_path, loop_config)
loop_config = load_loop_config(str(profile_path))
report = readiness(parent_config, loop_config, check_runtime=False)
print("Readiness:", report)
if not report["ready"]:
    raise RuntimeError("Resolve the data/configuration issues above before loading the model.")
schemas = json.loads((PROJECT_ROOT / "data/self_instruct/task_schemas.json").read_text())["tasks"]
subset = save_colab_subset(loop_config, schemas, RUN_DIR)
print("Fixed evaluation cases:", subset["total"], "Per task:", subset["per_task"])

saved_config = RUN_DIR / "colab_config.json"
if saved_config.exists():
    saved_profile = json.loads(saved_config.read_text())
    # Older notebooks allowed these execution settings to be chosen before the run.
    for section, keys in (("training", ("per_device_train_batch_size", "gradient_accumulation_steps")),
                          ("inference", ("batch_size",))):
        for key in keys:
            loop_config[section][key] = saved_profile[section][key]
    if saved_profile != loop_config:
        raise RuntimeError("This Drive run folder has different settings; keep the original settings or choose a new RUN_NAME.")
# Confirm Drive supports the actual journal write and local process-lock path.
probe = Store(RUN_DIR / "storage_probe")
with probe.lock():
    probe.append("probe.jsonl", {"drive_write_test": True})
(probe.directory / "probe.jsonl").unlink()
atomic_json(saved_config, loop_config)
print("Google Drive writes verified:", RUN_DIR)
print(f'{loop_config["max_rounds"]} rounds × {loop_config["scenarios_per_round"]} new pairs')
print(f'{loop_config["adaptive_pairs_per_round"]} adaptive pairs per round | '
      f'requested training batch {TRAIN_BATCH_SIZE} | inference batch {INFERENCE_BATCH_SIZE}')


## Load the student model

This cell loads Qwen3-8B and reports the assigned GPU. The run cell uses the teacher API to grade open-ended benchmark answers and generate training pairs.


In [ ]:
# Import Unsloth before Transformers, PEFT or TRL so its patches apply.
import unsloth
import torch
from distillation_loop.colab import UnslothStudentRuntime, l4_runtime_config
from distillation_loop.evaluation import require_training_versions

require_training_versions()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
runtime_config = l4_runtime_config(
    loop_config, train_batch_size=TRAIN_BATCH_SIZE, inference_batch_size=INFERENCE_BATCH_SIZE,
)
student_runtime = UnslothStudentRuntime.load(MODEL_NAME, runtime_config)
print("Unsloth adapter ready. Inference batch size:", INFERENCE_BATCH_SIZE)
torch.cuda.reset_peak_memory_stats()


## Start or resume training

Run this cell when you’re ready to begin. The teacher API budget is cumulative across sessions. On resume, press Enter to reuse the saved cap or enter a higher cap. Keep the same Drive folder and settings so the notebook can reuse saved progress.


In [ ]:
from distillation_loop.controller import run_loop
from self_instruct_pipeline.budget import APIBudget, BudgetExceeded
from self_instruct_pipeline.usage import export_usage

from google.colab import userdata
from self_instruct_pipeline.usage_cloud import SupabaseLedger, sync_usage_history

for name in ("SUPABASE_URL", "SUPABASE_SECRET_KEY"):
    if not os.environ.get(name):
        try:
            os.environ[name] = userdata.get(name).strip()
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
if SupabaseLedger.from_environment():
    print("Supabase usage sync enabled.", sync_usage_history(RUN_DIR))
else:
    print("API usage is recorded in Drive.")

budget_path = RUN_DIR / "api_budget" / "state.json"
saved_limits = json.loads(budget_path.read_text())["limits"] if budget_path.exists() else None
prompt = "Total teacher API budget in USD"
if saved_limits:
    prompt += f' [Enter to keep ${saved_limits["max_cost_usd"]}]'
budget_usd = input(prompt + ": ").strip()
if not budget_usd and saved_limits:
    budget_usd = saved_limits["max_cost_usd"]
if not budget_usd:
    raise ValueError("Enter a teacher API budget before starting.")
max_calls = max(10000, saved_limits["max_calls"]) if saved_limits else 10000
budget = APIBudget(RUN_DIR / "api_budget", max_calls=max_calls, max_cost_usd=budget_usd)

try:
    with budget.activate(allow_increase=True):
        result = run_loop(
            student_runtime, parent_config, loop_config,
        )
except BudgetExceeded as exc:
    print("Paused at teacher API budget:", exc)
    print("Rerun this cell with a higher cumulative cap to resume.")
else:
    print("Stop reason:", result["status"])
    print("Completed rounds:", result["completed_rounds"])
    print("Selected checkpoint:", result.get("selected_checkpoint", result["best_checkpoint"]))
    print("Best validation score:", result["best_report"]["macro_task_score"])
finally:
    print("Peak GPU memory reserved (GiB):", round(torch.cuda.max_memory_reserved() / 1024**3, 1))
    print("Google Drive run folder:", RUN_DIR)
    print("Usage:", export_usage(RUN_DIR))


In [ ]:
# Inspect persistent backups and progress without training or API calls.
for receipt_path in sorted((RUN_DIR / "drive_backups").glob("round_*.json")):
    receipt = json.loads(receipt_path.read_text())
    print(receipt["status"], receipt["archive"])
state_path = RUN_DIR / "loop_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    print("Completed rounds:", state["completed_rounds"], "Status:", state["status"])
    for entry in state["history"]:
        print("Round", entry["round"], "macro", entry["scores"]["macro_task_score"])


In [ ]:
# Retry only the latest Drive ZIP export if a backup operation was interrupted.
# This does not retrain the model or regenerate teacher data.
from distillation_loop.drive_backup import save_drive_adapter
completed = sorted(RUN_DIR.glob("round_*/training_complete.json"),
                   key=lambda path: int(path.parent.name.split("_")[-1]))
if completed:
    latest = completed[-1]
    checkpoint = json.loads(latest.read_text())["checkpoint"]
    save_drive_adapter(checkpoint, RUN_DIR, int(latest.parent.name.split("_")[-1]))
else:
    print("No completed round yet. Initial adapter/state may still be available on Drive.")
